# Oil Supply Chain Optimization — Analysis Notebook

**Author:** Portfolio project for Chevron Optimization Engineer application  
**Stack:** PuLP MILP · CBC solver (bundled) · Stochastic programming · Rolling horizon MPC

---

## 1. Problem Setup

This notebook walks through the complete analysis pipeline:

1. Build the Permian Basin → Gulf Coast network
2. Solve the multi-period MILP base case
3. Analyze crude grade routing and refinery crude diet constraints
4. Evaluate 9 deterministic scenarios
5. Run two-stage stochastic analysis (EVPI, VSS, VaR, CVaR)
6. Run 30-day rolling horizon simulation
7. Sensitivity / bottleneck analysis
8. Carbon–margin Pareto frontier

In [1]:
import sys, os
sys.path.insert(0, os.path.dirname(os.getcwd()))  # add osc/ to path

import copy
import warnings
import pandas as pd
import numpy as np
import plotly.io as pio

pio.renderers.default = 'notebook'
warnings.filterwarnings('ignore')

from data.generate_data import build_base_network, apply_disruption
from src.model.optimizer import MultiPeriodOptimizer
from src.model.supply_chain import NodeType
from src.viz.charts import *

print('Imports OK')

Imports OK


## 2. Network Overview

In [2]:
net = build_base_network(horizon=14)

# Print network summary
print(f"Planning horizon: {net.planning_horizon} days")
print(f"Nodes: {len(net.nodes)}")
print(f"  Wells: {len(net.get_nodes_by_type(NodeType.WELL))}")
print(f"  Storage: {len(net.get_nodes_by_type(NodeType.STORAGE))}")
print(f"  Refineries: {len(net.get_nodes_by_type(NodeType.REFINERY))}")
print(f"  Distribution: {len(net.get_nodes_by_type(NodeType.DISTRIBUTION))}")
print(f"  Demand: {len(net.get_nodes_by_type(NodeType.DEMAND))}")
print(f"Arcs: {len(net.arcs)}")
print(f"Grades: {net.grade_ids()}")
print(f"Ship-or-Pay contracts: {len(net.contracts)}")

print("\nCrude Grade Properties:")
for gid, g in net.grades.items():
    print(f"  {gid}: API={g.api_gravity}, S={g.sulfur_content}%, "
          f"diff={g.price_differential:+.2f} $/bbl, "
          f"CI={g.carbon_intensity} kg CO2e/bbl")

print("\nRefinery Crude Diets:")
for r in net.get_nodes_by_type(NodeType.REFINERY):
    d = r.crude_diet
    print(f"  {r.name}: API [{d.api_min}-{d.api_max}], S ≤ {d.sulfur_max}%")

Planning horizon: 14 days
Nodes: 18
  Wells: 5
  Storage: 3
  Refineries: 3
  Distribution: 2
  Demand: 5
Arcs: 34
Grades: ['WTI', 'WTS', 'HEAVY']
Ship-or-Pay contracts: 4

Crude Grade Properties:
  WTI: API=40.8, S=0.24%, diff=+0.00 $/bbl, CI=8.5 kg CO2e/bbl
  WTS: API=34.0, S=1.52%, diff=-3.20 $/bbl, CI=10.2 kg CO2e/bbl
  HEAVY: API=27.5, S=2.1%, diff=-7.40 $/bbl, CI=11.8 kg CO2e/bbl

Refinery Crude Diets:
  Houston Refinery (Light Sweet): API [35.0-45.0], S ≤ 0.5%
  Port Arthur Complex (Coking): API [24.0-43.0], S ≤ 2.8%
  Beaumont Complex (Mid-Range): API [30.0-44.0], S ≤ 1.2%


## 3. Base Case Optimization

In [ ]:
opt = MultiPeriodOptimizer(copy.deepcopy(net))
result = opt.solve()

T = result.planning_horizon
total_demand = sum(n.demand for n in net.get_nodes_by_type(NodeType.DEMAND)) * T
total_unmet  = sum(result.unmet_demand.values())
svc = (1 - total_unmet / max(total_demand, 1)) * 100
total_carbon = sum(result.carbon_by_period.values())

print(f"Status:          {result.status}")
print(f"Net Margin:      ${result.objective_value:>12,.0f}")
print(f"Gross Revenue:   ${result.revenue:>12,.0f}")
print(f"Transport:       ${result.transport_cost:>12,.0f}")
print(f"Holding:         ${result.holding_cost:>12,.0f}")
print(f"SOP Deficiency:  ${result.sop_cost:>12,.0f}")
print(f"Unmet Penalties: ${result.penalty_cost:>12,.0f}")
print(f"Service Level:   {svc:>11.1f}%")
print(f"Total Carbon:    {total_carbon:>10.1f} tCO2e")
print(f"Solver Time:     {result.solver_time:>11.2f}s")

In [ ]:
# Flow network map (Day 1)
network_flow_map(net, result, period=1).show()

In [ ]:
# P&L breakdown
cost_waterfall(result).show()

## 4. Crude Grade Routing Analysis

Key question: **does the optimizer respect crude diet constraints?**

- R1 Houston (light sweet, sulfur_max=0.50%) should receive almost no WTS/HEAVY
- R2 Port Arthur (coking unit, sulfur_max=2.80%) is the natural sink for sour grades

In [ ]:
grade_ids = net.grade_ids()
refineries = net.get_nodes_by_type(NodeType.REFINERY)

print("Average crude grade inflow per refinery (bbl/day):")
print(f"{'Refinery':<35} " + " ".join(f"{g:>10}" for g in grade_ids))
print("-" * 65)

for r in refineries:
    vals = []
    for g in grade_ids:
        avg_inflow = sum(
            result.flows_by_grade.get((i, r.id, t, g), 0)
            for i in net.nodes
            for t in range(1, T + 1)
            if (i, r.id) in net.arc_index
        ) / T
        vals.append(avg_inflow)
    print(f"{r.name:<35} " + " ".join(f"{v:>10,.0f}" for v in vals))

In [ ]:
# Refinery grade mix chart
refinery_grade_mix(net, result, period=1).show()

In [ ]:
# Storage inventory trajectories
inventory_profile(net, result).show()

## 5. Arc Utilization & Bottleneck Identification

In [ ]:
arc_utilization_chart(net, result).show()

In [ ]:
from src.analysis.sensitivity import identify_bottlenecks

bottlenecks = identify_bottlenecks(net, result)

print(f"{'Arc':<50} {'Avg Util':>8} {'Max Util':>8} {'Shadow $/bbl-day':>16} {'Periods@cap':>12}")
print("-" * 100)
for b in bottlenecks:
    print(f"{b.arc_label:<50} {b.avg_utilization:>7.1%} {b.max_utilization:>7.1%} "
          f"${b.shadow_value_estimate:>14.2f} {b.periods_at_capacity:>12}")

## 6. Scenario Analysis

In [ ]:
from src.analysis.scenario import ScenarioRunner, build_standard_scenarios

runner = ScenarioRunner(copy.deepcopy(net))
runner._base_result = result
sc_results = runner.run_all(build_standard_scenarios(net))

rows = []
for s in sc_results:
    T_s = s.result.planning_horizon
    unmet_s = sum(s.result.unmet_demand.values())
    dem_s = sum(n.demand for n in net.get_nodes_by_type(NodeType.DEMAND)) * T_s
    svc_s = (1 - unmet_s / max(dem_s, 1)) * 100
    rows.append({
        'Scenario': s.scenario_name,
        'Category': s.category,
        'Net Margin': s.result.objective_value,
        'Delta': s.delta_objective,
        'Service %': svc_s,
        'Carbon t/d': sum(s.result.carbon_by_period.values()) / T_s,
    })

df_sc = pd.DataFrame(rows).sort_values('Delta')
df_sc['Net Margin'] = df_sc['Net Margin'].map('${:,.0f}'.format)
df_sc['Delta'] = df_sc['Delta'].map('${:+,.0f}'.format)
df_sc['Service %'] = df_sc['Service %'].map('{:.1f}%'.format)
df_sc['Carbon t/d'] = df_sc['Carbon t/d'].map('{:.0f}'.format)
df_sc

In [ ]:
base_obj = result.objective_value
names = [s.scenario_name for s in sc_results]
objs = [s.result.objective_value for s in sc_results]
cats = [s.category for s in sc_results]

scenario_comparison(names, objs, base_obj, cats).show()

## 7. Sensitivity Analysis (Tornado Chart)

In [ ]:
from src.analysis.sensitivity import run_sensitivity

entries = run_sensitivity(copy.deepcopy(net), result, delta_pct=0.10)

print(f"{'Parameter':<40} {'Swing Up':>12} {'Swing Down':>12} {'Total Swing':>12}")
print("-" * 80)
for e in entries[:8]:
    print(f"{e.parameter:<40} ${e.swing_up:>10,.0f} ${e.swing_down:>10,.0f} ${e.total_swing:>10,.0f}")

In [ ]:
tornado_chart(entries, top_n=8).show()

## 8. Two-Stage Stochastic Programming

**Key question:** What is the value of optimizing explicitly under crack spread uncertainty?

- **EVPI** = WS − RP: maximum worth paying for a perfect price forecast
- **VSS** = RP − EEV: gain from stochastic vs. deterministic planning

Interpretation: if EVPI is large relative to the cost of a price forecasting service, it's worth buying.

In [ ]:
from src.model.stochastic import run_stochastic_analysis

sr = run_stochastic_analysis(
    copy.deepcopy(net),
    n_scenarios=10,
    use_extensive_form=False,   # fast mode; set True for exact EF
)

print(f"RP   (Recourse Problem)      : ${sr.rp:>12,.0f}")
print(f"WS   (Wait and See)          : ${sr.ws:>12,.0f}")
print(f"EEV  (Mean-Value Evaluated)  : ${sr.eev:>12,.0f}")
print(f"EVPI (WS - RP)               : ${sr.evpi:>12,.0f}")
print(f"VSS  (RP - EEV)              : ${sr.vss:>12,.0f}")
print()
print(f"VaR 95%  : ${sr.risk.var_95:>10,.0f}")
print(f"CVaR 95% : ${sr.risk.cvar_95:>10,.0f}")
print(f"VaR 99%  : ${sr.risk.var_99:>10,.0f}")
print(f"Std Dev  : ${sr.risk.std_obj:>10,.0f}")
print(f"Solve time: {sr.solver_time:.1f}s")

In [ ]:
evpi_vss_table(sr).show()
stochastic_fan(sr).show()

## 9. Rolling Horizon (MPC) Simulation

Simulates 30 days of actual planning loop:
1. Solve 7-day lookahead MILP
2. Execute Day 1 only
3. Inject noise (production ±5%, demand ±8%, crack spread daily vol)
4. Update state, re-plan

**Replanning value** = realized RH margin − static plan margin  
This is the quantified benefit of adaptive re-optimization vs. holding the initial plan.

In [ ]:
from src.analysis.rolling_horizon import RollingHorizonOptimizer
from src.viz.rolling_charts import rolling_margin_chart, service_level_trend

rh = RollingHorizonOptimizer(
    copy.deepcopy(net),
    rolling_horizon=7,
    simulation_days=30,
    replan_trigger='always',
    noise_level=0.05,
)

rh_result = rh.run()

print(f"Total Realized Margin : ${rh_result.total_realized_margin:>12,.0f}")
print(f"Static Plan Margin    : ${rh_result.static_plan_margin:>12,.0f}")
print(f"Replanning Value      : ${rh_result.replanning_value:>12,.0f}")
print(f"Avg Service Level     : {rh_result.avg_service_level:>11.1%}")
print(f"Plan-Execution Gap    : ${rh_result.plan_execution_gap:>12,.0f}/day")

In [ ]:
rolling_margin_chart(rh_result.execution_log).show()
service_level_trend(rh_result.execution_log).show()

## 10. Carbon–Margin Pareto Frontier

What is the implicit carbon price at each level of constraint stringency?

In [ ]:
from data.generate_data import apply_carbon_cap

base_carbon = sum(result.carbon_by_period.values()) / T
budgets = [base_carbon * f for f in [1.00, 0.92, 0.85, 0.78, 0.70, 0.62, 0.55]]

pareto_rows = []
for budget in budgets:
    n_copy = copy.deepcopy(net)
    apply_carbon_cap(n_copy, budget)
    r_copy = MultiPeriodOptimizer(n_copy).solve()
    c_actual = sum(r_copy.carbon_by_period.values()) / T
    margin_cost = result.objective_value - r_copy.objective_value
    delta_carbon = (base_carbon - c_actual) * T
    implicit_price = margin_cost / max(delta_carbon, 1)
    pareto_rows.append({
        'Budget (t/d)': f'{budget:.0f}',
        'Realized Carbon (t/d)': f'{c_actual:.1f}',
        'Net Margin ($)': f'${r_copy.objective_value:,.0f}',
        'Margin Cost ($)': f'${margin_cost:,.0f}',
        'Implicit Carbon Price ($/t)': f'${implicit_price:.0f}',
    })

pd.DataFrame(pareto_rows)

In [ ]:
import plotly.graph_objects as go

x_c = [float(r['Realized Carbon (t/d)']) for r in pareto_rows]
y_m = [float(r['Net Margin ($)'].replace('$','').replace(',','')) for r in pareto_rows]

fig = go.Figure(go.Scatter(
    x=x_c, y=y_m,
    mode='lines+markers+text',
    text=[r['Budget (t/d)'] + ' t/d' for r in pareto_rows],
    textposition='top center',
    line=dict(color='#58a6ff', width=2),
    marker=dict(size=8),
))
fig.update_layout(
    title='Carbon–Margin Pareto Frontier',
    xaxis_title='Avg Daily Carbon (tCO2e/day)',
    yaxis_title='Net Margin ($)',
    paper_bgcolor='#0d1117', plot_bgcolor='#0d1117',
    font=dict(color='#c9d1d9'),
    height=400,
)
fig.show()

## 11. Demand Forecasting Integration

In [ ]:
from src.analysis.demand_forecast import (
    DemandForecaster, generate_synthetic_history, MARKET_PROFILES
)
from src.viz.rolling_charts import forecast_chart

# Build forecaster on 180 days of synthetic history
forecaster = DemandForecaster()
forecaster.fit_from_defaults(list(MARKET_PROFILES.keys()), n_history_days=180)
forecasts = forecaster.forecast(horizon=14)

# Summary
for row in forecaster.forecast_summary():
    print(row)

# Fan chart for Chicago
hist = generate_synthetic_history('M1', n_days=180)
forecast_chart('Chicago', forecasts['M1'], hist).show()

In [ ]:
# Integrate forecasts into optimizer
from src.analysis.demand_forecast import build_network_with_forecasts

demand_params = forecaster.predict_demand_params(horizon=14)
net_forecast = build_network_with_forecasts(net, demand_params, horizon=14)

result_forecast = MultiPeriodOptimizer(net_forecast).solve()

print(f"Static demand net margin:   ${result.objective_value:>12,.0f}")
print(f"Forecast demand net margin: ${result_forecast.objective_value:>12,.0f}")
print(f"Delta:                      ${result_forecast.objective_value - result.objective_value:>+12,.0f}")